## 4. Weak SINDy for the EM density

Weak SINDy uses integration by parts instead of computing numerical derivative at query points to reduce the affect caused by noise. More precisely, when we implement the usuay SINDy idea, we only have density data $\rho$ at qrid points $(x_i,t_j)$, so in order to get data for derivative, we apply the finite difference method at these query points. For example, suppose the noisy density data $\hat{\rho}_k = \rho_k +\epsilon_k$ where $\rho_k$ is the exact density data at the $k$th query point and $\epsilon_k \sim N(0,\sigma^2)$ is the white noise. If we use the central difference for the first derivative
$$\partial_x \hat{\rho} = \frac{\hat{\rho}_{k+1} - \hat{\rho}_{k-1}}{2\Delta x},$$
the noise data is 
$$\frac{\epsilon_{k+1} - \epsilon_{k-1}}{2\Delta x}$$
which has variance $\frac{\sigma^2}{2\Delta x^2}$.
Similarly, the second order difference method for the second derivative has variance $$\frac{6\sigma^2}{\Delta x^4}.$$
Consequently, although a smaller grid spacing reduces the truncation error of the finite-difference approximation, it amplifies the variance of the differentiated noise. This helps explain why refining the spatial or temporal grid does not necessarily improve pointwise SINDy identification when the density data are noisy.

On the other hand, write the OU Fokker--Planck equation in conservative form as

$$
\rho_t = \kappa\,\partial_x q + D\rho_{xx},
\qquad q(x,t)=x\rho(x,t).
$$

For every randomly sampled space--time subdomain $\Omega_k$, weak SINDy uses a test function $\phi_k$ whose required lower-order derivatives and $\phi_k$ itself vanish on the boundary of the local subdomain, so all boundary terms produced by integration by parts are zero. Integration by parts gives

$$
-\int_{\Omega_k}\rho (\phi_k)_t \,dxdt
=
-\kappa \int_{\Omega_k}q\,(\phi_k)_x \, dxdt
+
D \int_{\Omega_k}\rho (\phi_k)_{xx}\,dxdt.
$$

Therefore, Weak SINDy does not require pointwise numerical approximations of $\rho_t$, $q_x$ and $\rho_{xx}$. Instead, the derivatives are transferred to the known smooth test function. The observed density enters only through weighted local integrals, so the noise is not differentiated and its positive and negative contributions can partially cancel.

However, the subdomain size introduces a bias–variance trade-off. A larger subdomain averages over more observations and therefore suppresses random noise more effectively. If it is too large, however, the weak measurements may over-average localized spatial and temporal variations. This can reduce the contrast between different candidate terms and make the PDE coefficients less identifiable.




In [1]:
# ============================================================
# Imports
# ============================================================

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pysindy as ps

from IPython.display import display

import ou_em

importlib.reload(ou_em)
solve_ou_em = ou_em.solve_ou_em

In [2]:
# ============================================================
# Shared OU parameters and helper functions
# ============================================================

kappa = 1.0
D = 1.0

# Non-stationary Gaussian initial distribution
m0 = -2.0
s0 = 0.5

# Space-time domain
x_L = -5.0
x_R = 5.0
T_final = 3.0


def exact_ou_moments(t_grid, kappa, D, m0, s0):
    """Return the exact OU mean, variance and their time derivatives."""

    mean = m0 * np.exp(-kappa * t_grid)

    variance = (
        s0**2 * np.exp(-2.0 * kappa * t_grid)
        + (D / kappa) * (
            1.0 - np.exp(-2.0 * kappa * t_grid)
        )
    )

    mean_t = -kappa * mean
    variance_t = -2.0 * kappa * variance + 2.0 * D

    return mean, variance, mean_t, variance_t


def exact_ou_density(x_grid, t_grid, kappa, D, m0, s0):
    """Exact density of dX_t = -kappa X_t dt + sqrt(2D) dW_t."""

    mean, variance, _, _ = exact_ou_moments(
        t_grid,
        kappa,
        D,
        m0,
        s0,
    )

    return (
        np.exp(
            -0.5 * (x_grid - mean)**2 / variance
        )
        / np.sqrt(2.0 * np.pi * variance)
    )


def exact_ou_time_derivative(x_grid, t_grid, kappa, D, m0, s0):
    """Analytical time derivative of the exact OU density."""

    mean, variance, mean_t, variance_t = exact_ou_moments(
        t_grid,
        kappa,
        D,
        m0,
        s0,
    )

    density = exact_ou_density(
        x_grid,
        t_grid,
        kappa,
        D,
        m0,
        s0,
    )

    return density * (
        mean_t * (x_grid - mean) / variance
        + variance_t
        / (2.0 * variance**2)
        * (
            (x_grid - mean)**2 - variance
        )
    )


def relative_l2_error(approximation, reference):
    """Relative Euclidean L2 error after flattening both arrays."""

    approximation = np.asarray(approximation).ravel()
    reference = np.asarray(reference).ravel()

    reference_norm = np.linalg.norm(reference)

    if reference_norm == 0.0:
        raise ZeroDivisionError(
            "The reference array has zero L2 norm."
        )

    return (
        np.linalg.norm(approximation - reference)
        / reference_norm
    )


def print_identified_equation(feature_names, coefficients, tolerance=1e-10):
    """Print a readable sparse PDE from identified coefficients."""

    identified_terms = []

    for name, coefficient in zip(feature_names, coefficients):
        if abs(coefficient) > tolerance:
            identified_terms.append(
                f"({coefficient:+.6f}) {name}"
            )

    equation = " ".join(identified_terms)

    print("True Fokker–Planck equation:")
    print(
        f"rho_t = {kappa:.6f} d_x(x rho)"
        f" + {D:.6f} rho_xx"
    )

    print()
    print("Identified Fokker–Planck equation:")
    print(f"rho_t = {equation}")


def plot_error_heatmap(t_grid, x_grid, error_field, title, colorbar_label="Error"):
    """Plot a signed error field with symmetric colour limits."""

    max_abs_error = np.max(np.abs(error_field))
    max_abs_error = max(max_abs_error, np.finfo(float).eps)

    fig, ax = plt.subplots(figsize=(8, 5))

    plot = ax.pcolormesh(
        t_grid,
        x_grid,
        error_field,
        shading="auto",
        vmin=-max_abs_error,
        vmax=max_abs_error,
        cmap="coolwarm",
    )

    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"$x$")
    ax.set_title(title)

    fig.colorbar(
        plot,
        ax=ax,
        label=colorbar_label,
    )

    plt.tight_layout()
    plt.show()


In [3]:
# ============================================================
# 3.2 EM resolution and saved data times
# ============================================================

# Number of spatial intervals used by the histogram density
Nx_em = 50      # 50 100 200 300

# Number of saved time intervals supplied to SINDy
Nt_data_em = 75     # 75 150 300 600

# Number of internal Euler-Maruyama steps
Nt_internal_em = 1500   # 1500 3000 6000 12000

# Number of Monte Carlo sample paths
num_paths_em = 60000   # 5000 60000 120000

if Nt_internal_em % Nt_data_em != 0:
    raise ValueError(
        "Nt_internal_em must be divisible by Nt_data_em so every "
        "saved time lies on the internal EM time grid."
    )

save_times_em = np.linspace(
    0.0,
    T_final,
    Nt_data_em + 1,
)

print("Number of saved times:       ", save_times_em.size)
print("Internal EM time step:       ", T_final / Nt_internal_em)
print("Saved-density time spacing:  ", T_final / Nt_data_em)
print("Number of Monte Carlo paths: ", num_paths_em)


Number of saved times:        76
Internal EM time step:        0.002
Saved-density time spacing:   0.04
Number of Monte Carlo paths:  60000


In [4]:
# ============================================================
# 3.3 Generate EM density snapshots
# ============================================================

em_solution = solve_ou_em(
    num_paths=num_paths_em,
    N_em=Nt_internal_em,
    Nx=Nx_em,
    k=kappa,
    D=D,
    x_L=x_L,
    x_R=x_R,
    m0=m0,
    s0=s0,
    T=T_final,
    save_times=save_times_em,
    seed=319,
    store_particles=False,
)


In [5]:
# ============================================================
# 3.4 Convert the EM snapshots into rho_em(x,t)
# ============================================================

x_em = np.asarray(
    em_solution["x"],
    dtype=float,
)

t_em = np.array(
    sorted(em_solution["snapshots"].keys()),
    dtype=float,
)

rho_em = np.column_stack([
    em_solution["snapshots"][saved_time]
    for saved_time in t_em
])

dx_em = float(em_solution["dx"])
dt_internal_em = float(em_solution["dt_em"])
dt_data_em = float(t_em[1] - t_em[0])

X_em, Time_em = np.meshgrid(
    x_em,
    t_em,
    indexing="ij",
)

if rho_em.shape != X_em.shape:
    raise ValueError(
        f"rho_em has shape {rho_em.shape}, "
        f"but the EM coordinate grid has shape {X_em.shape}."
    )

# Independent analytical references evaluated on the EM grid
rho_exact_em_grid = exact_ou_density(
    X_em,
    Time_em,
    kappa,
    D,
    m0,
    s0,
)

rho_t_exact_em_grid = exact_ou_time_derivative(
    X_em,
    Time_em,
    kappa,
    D,
    m0,
    s0,
)

print("rho_em shape:", rho_em.shape)
print("x spacing:    ", dx_em)
print("saved dt:     ", dt_data_em)


rho_em shape: (51, 76)
x spacing:     0.20000000000000018
saved dt:      0.04


## Lib 1

In [6]:
# ============================================================
# 4.1 Space-time grid, state field and conservative-flux field
# ============================================================

# WeakPDELibrary expects the final axis of the grid to contain
# the coordinates.  The resulting shape is (Nx + 1, Nt + 1, 2).
XT_em = np.stack(
    (X_em, Time_em),
    axis=-1,
)

# One dynamical state: rho(x,t)
rho_state_weak_em = rho_em[..., np.newaxis]

# Four auxiliary flux fields used only on the RHS
candidate_control_weak_em = np.stack(
    [
        X_em * rho_em,          # x rho
        X_em**2 * rho_em,       # x^2 rho
        X_em * rho_em**2,       # x rho^2
        X_em**2 * rho_em**2,    # x^2 rho^2
    ],
    axis=-1,
)


print("XT_em shape:                 ", XT_em.shape)
print("rho_state_weak_em shape:     ", rho_state_weak_em.shape)
print("candidate_control_weak_em shape:", candidate_control_weak_em.shape)


XT_em shape:                  (51, 76, 2)
rho_state_weak_em shape:      (51, 76, 1)
candidate_control_weak_em shape: (51, 76, 4)


In [7]:
# ============================================================
# 4.2 Construct the weak PDE library
# ============================================================

# K is the number of randomly sampled space-time subdomains.
K_weak_em = 1000    # 250 500 1000 2000

# H_xt contains the HALF-width of each subdomain in the
# spatial and temporal directions.  These values reproduce
# the library default L_xt / 20 explicitly.
H_xt_weak_em = np.array([
    (x_em[-1] - x_em[0]) / 20.0,    # /40  /20  /10
    (t_em[-1] - t_em[0]) / 20.0,
])

# The random subdomains are selected when WeakPDELibrary is
# constructed, so set the seed immediately before construction.
np.random.seed(319)

weak_library_em = ps.WeakPDELibrary(
    # IdentityLibrary creates the five zero-order fields.
    # WeakPDELibrary additionally integrates their first and
    # second spatial derivatives on the same K subdomains.
    function_library=ps.IdentityLibrary(),
    derivative_order=2,
    spatiotemporal_grid=XT_em,
    include_bias=False,
    include_interaction=False,
    K=K_weak_em,
    H_xt=H_xt_weak_em,
    p=4,
)

print("K:                    ", K_weak_em)
print("H_xt (half-widths):   ", H_xt_weak_em)
print("Full subdomain widths:", 2.0 * H_xt_weak_em)


K:                     1000
H_xt (half-widths):    [0.5  0.15]
Full subdomain widths: [1.  0.3]


In [8]:
# ============================================================
# 4.3 Fit the weak-form sparse model
# ============================================================

# Assemble the five fields on which the weak integrals are evaluated.
weak_input_fields_em = np.concatenate(
    [rho_state_weak_em, candidate_control_weak_em],
    axis=-1,
)

weak_input_names_em = [
    "rho",
    "xrho",
    "x2rho",
    "xrho2",
    "x2rho2",
]

# Build the raw weak matrix directly from the library. This happens
# before any regression or column normalization.
weak_library_em.fit(weak_input_fields_em)
weak_full_matrix_em = np.asarray(
    weak_library_em.transform(weak_input_fields_em)
)
weak_full_feature_names_em = weak_library_em.get_feature_names(
    weak_input_names_em
)

# Select exactly the intended conservative candidates BEFORE STLSQ.
weak_feature_names_em = [
    "xrho_1",
    "x2rho_1",
    "xrho2_1",
    "x2rho2_1",
    "rho_11",
]

weak_selected_indices_em = [
    weak_full_feature_names_em.index(name)
    for name in weak_feature_names_em
]

weak_regression_matrix_em = weak_full_matrix_em[
    :, weak_selected_indices_em
]

# The weak LHS is built by the same library on the same K subdomains.
weak_target_em = weak_library_em.convert_u_dot_integral(
    rho_state_weak_em
)

optimizer_weak_em = ps.STLSQ(
    threshold=0.1,
    alpha=1e-10,
    max_iter=100,
    normalize_columns=True,
)

optimizer_weak_em.fit(
    weak_regression_matrix_em,
    weak_target_em,
)

weak_coefficients_em = optimizer_weak_em.coef_.reshape(-1)

print("Full weak matrix shape:      ", weak_full_matrix_em.shape)
print("Regression matrix shape:     ", weak_regression_matrix_em.shape)
print("Weak target shape:            ", weak_target_em.shape)
print("Number of fitted equations:  ", optimizer_weak_em.coef_.shape[0])
print("Regression candidate terms:")
for name in weak_feature_names_em:
    print("  ", name)


Full weak matrix shape:       (1000, 15)
Regression matrix shape:      (1000, 5)
Weak target shape:             (1000, 1)
Number of fitted equations:   1
Regression candidate terms:
   xrho_1
   x2rho_1
   xrho2_1
   x2rho2_1
   rho_11


In [9]:
# ============================================================
# 4.4 Coefficient table and identified conservative PDE
# ============================================================

# Internal PySINDy names retain _1 and _11. The table and printed
# equation use readable x-derivative names instead.
weak_display_name_em = {
    "xrho_1": r"$(x \rho)_x$",
    "x2rho_1": r"$(x^2 \rho)_x$",
    "xrho2_1": r"$(x \rho^2)_x$",
    "x2rho2_1": r"$(x^2 \rho^2)_x$",
    "rho_11": r"$\rho_{xx}$",
}

readable_feature_names_em = [
    weak_display_name_em[name]
    for name in weak_feature_names_em
]

weak_true_map_em = {
    "xrho_1": kappa,
    "rho_11": D,
}

weak_true_coefficients_em = np.array([
    weak_true_map_em.get(name, 0.0)
    for name in weak_feature_names_em
])

weak_coefficient_table_em = pd.DataFrame({
    "Candidate term": readable_feature_names_em,
    "Identified coefficient": weak_coefficients_em,
    "True coefficient": weak_true_coefficients_em,
    "Absolute error": np.abs(
        weak_coefficients_em - weak_true_coefficients_em
    ),
})

display(
    weak_coefficient_table_em.style.format({
        "Identified coefficient": "{:.6f}",
        "True coefficient": "{:.6f}",
        "Absolute error": "{:.3e}",
    }).set_caption("Weak-SINDy coefficients from EM histogram density")
)

print_identified_equation(
    readable_feature_names_em,
    weak_coefficients_em,
)


,Candidate term,Identified coefficient,True coefficient,Absolute error
0,$(x \rho)_x$,0.926690,1.000000,7.331e-02
1,$(x^2 \rho)_x$,0.000000,0.000000,0.000e+00
2,$(x \rho^2)_x$,0.000000,0.000000,0.000e+00
3,$(x^2 \rho^2)_x$,0.000000,0.000000,0.000e+00
4,$\rho_{xx}$,0.896128,1.000000,1.039e-01


True Fokker–Planck equation:
rho_t = 1.000000 d_x(x rho) + 1.000000 rho_xx

Identified Fokker–Planck equation:
rho_t = (+0.926690) $(x \rho)_x$ (+0.896128) $\rho_{xx}$


In [10]:
# ============================================================
# 4.5 Weak-regression residual
# ============================================================

# Predict directly in the same K-dimensional weak system.
weak_prediction_em = (
    weak_regression_matrix_em
    @ weak_coefficients_em.reshape(-1, 1)
)

weak_relative_residual_em = relative_l2_error(
    weak_prediction_em,
    weak_target_em,
)

print("Weak target shape:     ", weak_target_em.shape)
print("Weak prediction shape: ", weak_prediction_em.shape)
print(
    "Relative L2 residual in weak regression:",
    f"{weak_relative_residual_em:.6e}",
)

# This residual lives in the K-dimensional weak-integral system.
# It is not a pointwise heatmap error on the original (x,t) grid.


Weak target shape:      (1000, 1)
Weak prediction shape:  (1000, 1)
Relative L2 residual in weak regression: 3.034942e-01


## Lib 2

In [11]:
# ============================================================
# 4.1 Space-time grid, state field and conservative-flux field
# ============================================================

# WeakPDELibrary expects the final axis of the grid to contain
# the coordinates.  The resulting shape is (Nx + 1, Nt + 1, 2).
XT_em = np.stack(
    (X_em, Time_em),
    axis=-1,
)

# One dynamical state: rho(x,t)
rho_state_weak_em = rho_em[..., np.newaxis]

# Four auxiliary flux fields used only on the RHS
candidate_control_weak_em = np.stack(
    [
        X_em * rho_em,          # x rho
        X_em**2 * rho_em,       # x^2 rho
        X_em * rho_em**2,       # x rho^2
        X_em**2 * rho_em**2,    # x^2 rho^2
    ],
    axis=-1,
)


print("XT_em shape:                 ", XT_em.shape)
print("rho_state_weak_em shape:     ", rho_state_weak_em.shape)
print("candidate_control_weak_em shape:", candidate_control_weak_em.shape)


XT_em shape:                  (51, 76, 2)
rho_state_weak_em shape:      (51, 76, 1)
candidate_control_weak_em shape: (51, 76, 4)


In [12]:
# ============================================================
# 4.2 Construct the weak PDE library
# ============================================================

# K is the number of randomly sampled space-time subdomains.
K_weak_em = 1000

# H_xt contains the HALF-width of each subdomain in the
# spatial and temporal directions.  These values reproduce
# the library default L_xt / 20 explicitly.
H_xt_weak_em = np.array([
    (x_em[-1] - x_em[0]) / 20.0,
    (t_em[-1] - t_em[0]) / 20.0,
])

# The random subdomains are selected when WeakPDELibrary is
# constructed, so set the seed immediately before construction.
np.random.seed(319)

weak_library_em = ps.WeakPDELibrary(
    # IdentityLibrary creates the five zero-order fields.
    # WeakPDELibrary additionally integrates their first and
    # second spatial derivatives on the same K subdomains.
    function_library=ps.IdentityLibrary(),
    derivative_order=2,
    spatiotemporal_grid=XT_em,
    include_bias=False,
    include_interaction=False,
    K=K_weak_em,
    H_xt=H_xt_weak_em,
    p=4,
)

print("K:                    ", K_weak_em)
print("H_xt (half-widths):   ", H_xt_weak_em)
print("Full subdomain widths:", 2.0 * H_xt_weak_em)


K:                     1000
H_xt (half-widths):    [0.5  0.15]
Full subdomain widths: [1.  0.3]


In [13]:
# ============================================================
# 4.3 Fit the weak-form sparse model
# ============================================================

# Assemble the five fields on which the weak integrals are evaluated.
weak_input_fields_em = np.concatenate(
    [rho_state_weak_em, candidate_control_weak_em],
    axis=-1,
)

weak_input_names_em = [
    "rho",
    "xrho",
    "x2rho",
    "xrho2",
    "x2rho2",
]

# Build the raw weak matrix directly from the library. This happens
# before any regression or column normalization.
weak_library_em.fit(weak_input_fields_em)
weak_full_matrix_em = np.asarray(
    weak_library_em.transform(weak_input_fields_em)
)
weak_full_feature_names_em = weak_library_em.get_feature_names(
    weak_input_names_em
)

# Keep the complete 15-column library: for each of the five
# input fields, retain the zero-, first- and second-order terms.
weak_feature_names_em = list(weak_full_feature_names_em)
weak_regression_matrix_em = weak_full_matrix_em

# The weak LHS is built by the same library on the same K subdomains.
weak_target_em = weak_library_em.convert_u_dot_integral(
    rho_state_weak_em
)

optimizer_weak_em = ps.STLSQ(
    threshold=0.1,
    alpha=1e-10,
    max_iter=100,
    normalize_columns=True,
)

optimizer_weak_em.fit(
    weak_regression_matrix_em,
    weak_target_em,
)

weak_coefficients_em = optimizer_weak_em.coef_.reshape(-1)

print("Full weak matrix shape:      ", weak_full_matrix_em.shape)
print("Regression matrix shape:     ", weak_regression_matrix_em.shape)
print("Weak target shape:            ", weak_target_em.shape)
print("Number of fitted equations:  ", optimizer_weak_em.coef_.shape[0])
print("Regression candidate terms:")
for name in weak_feature_names_em:
    print("  ", name)


Full weak matrix shape:       (1000, 15)
Regression matrix shape:      (1000, 15)
Weak target shape:             (1000, 1)
Number of fitted equations:   1
Regression candidate terms:
   rho
   xrho
   x2rho
   xrho2
   x2rho2
   rho_1
   xrho_1
   x2rho_1
   xrho2_1
   x2rho2_1
   rho_11
   xrho_11
   x2rho_11
   xrho2_11
   x2rho2_11


In [14]:
# ============================================================
# 4.4 Coefficient table and identified PDE
# ============================================================

# Internal PySINDy names retain _1 and _11. Use readable names
# for all 15 zero-, first- and second-order candidates.
weak_display_name_em = {
    "rho": r"$\rho$",
    "xrho": r"$x \rho$",
    "x2rho": r"x$^2 \rho$",
    "xrho2": r"$x \rho^2$",
    "x2rho2": r"$x^2 \rho^2$",
    "rho_1": r"$\rho_x$",
    "xrho_1": r"$(x \rho)_x$",
    "x2rho_1": r"$(x^2 \rho)_x$",
    "xrho2_1": r"$(x \rho^2)_x$",
    "x2rho2_1": r"$(x^2 \rho^2)_x$",
    "rho_11": r"$\rho_{xx}$",
    "xrho_11": r"$(x \rho)_{xx}$",
    "x2rho_11": r"$(x^2 \rho)_{xx}$",
    "xrho2_11": r"$(x \rho^2)_{xx}$",
    "x2rho2_11": r"$(x^2 \rho^2)_{xx}$",
}

readable_feature_names_em = [
    weak_display_name_em[name]
    for name in weak_feature_names_em
]

weak_true_map_em = {
    "xrho_1": kappa,
    "rho_11": D,
}

weak_true_coefficients_em = np.array([
    weak_true_map_em.get(name, 0.0)
    for name in weak_feature_names_em
])

weak_coefficient_table_em = pd.DataFrame({
    "Candidate term": readable_feature_names_em,
    "Identified coefficient": weak_coefficients_em,
    "True coefficient": weak_true_coefficients_em,
    "Absolute error": np.abs(
        weak_coefficients_em - weak_true_coefficients_em
    ),
})

display(
    weak_coefficient_table_em.style.format({
        "Identified coefficient": "{:.6f}",
        "True coefficient": "{:.6f}",
        "Absolute error": "{:.3e}",
    }).set_caption("Weak-SINDy coefficients from EM histogram density")
)

print_identified_equation(
    readable_feature_names_em,
    weak_coefficients_em,
)


,Candidate term,Identified coefficient,True coefficient,Absolute error
0,$\rho$,0.000000,0.000000,0.000e+00
1,$x \rho$,-0.834409,0.000000,8.344e-01
2,x$^2 \rho$,0.000000,0.000000,0.000e+00
3,$x \rho^2$,2.462583,0.000000,2.463e+00
4,$x^2 \rho^2$,0.000000,0.000000,0.000e+00
5,$\rho_x$,-0.995782,0.000000,9.958e-01
6,$(x \rho)_x$,0.000000,1.000000,1.000e+00
7,$(x^2 \rho)_x$,0.000000,0.000000,0.000e+00
8,$(x \rho^2)_x$,0.000000,0.000000,0.000e+00
9,$(x^2 \rho^2)_x$,0.000000,0.000000,0.000e+00


True Fokker–Planck equation:
rho_t = 1.000000 d_x(x rho) + 1.000000 rho_xx

Identified Fokker–Planck equation:
rho_t = (-0.834409) $x \rho$ (+2.462583) $x \rho^2$ (-0.995782) $\rho_x$


In [15]:
# ============================================================
# 4.5 Weak-regression residual
# ============================================================

# Predict directly in the same K-dimensional weak system.
weak_prediction_em = (
    weak_regression_matrix_em
    @ weak_coefficients_em.reshape(-1, 1)
)

weak_relative_residual_em = relative_l2_error(
    weak_prediction_em,
    weak_target_em,
)

print("Weak target shape:     ", weak_target_em.shape)
print("Weak prediction shape: ", weak_prediction_em.shape)
print(
    "Relative L2 residual in weak regression:",
    f"{weak_relative_residual_em:.6e}",
)

# This residual lives in the K-dimensional weak-integral system.
# It is not a pointwise heatmap error on the original (x,t) grid.


Weak target shape:      (1000, 1)
Weak prediction shape:  (1000, 1)
Relative L2 residual in weak regression: 4.867407e-01
